# Convolutional Neural Network (CNN) with PyTorch

A CNN learns spatial features with convolutional filters instead of treating
every input pixel independently. This notebook trains a small two-layer CNN on
the sklearn digits dataset (1797 8x8 grayscale images) and reports train/test
accuracy.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

# Load digits: 8x8 images, 10 classes. Add a channel dim -> (N, 1, 8, 8).
digits = load_digits()
X = digits.images[:, None, :, :].astype(np.float32) / 16.0  # scale pixels to [0, 1]
y = digits.target.astype(np.int64)
print(f"data: {X.shape}, labels: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
                          batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)),
                         batch_size=256)

In [ ]:
class SmallCNN(nn.Module):
    """Two-layer CNN: conv -> relu -> maxpool -> conv -> relu -> maxpool -> fc."""

    def __init__(self, num_classes=10):
        super(SmallCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        # 8x8 -> pool -> 4x4 -> pool -> 2x2, with 32 channels
        self.fc = nn.Linear(32 * 2 * 2, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        return self.fc(x)


model = SmallCNN()
print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"epoch {epoch + 1}/{num_epochs}, loss: {running_loss / len(train_loader):.4f}")

In [ ]:
def accuracy(loader):
    """Fraction of correctly classified samples in a data loader."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            correct += (model(images).argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return correct / total


print(f"train accuracy: {accuracy(train_loader):.4f}")
print(f"test accuracy:  {accuracy(test_loader):.4f}")